In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
df=pd.read_csv('dengue.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)
df.isnull().sum()

In [ ]:
#for years 2019,20,21
district_monthly_avg_cases = (
    df.groupby(['District', 'Year'])['Cases']
      .mean()
      .reset_index())


district_monthly_avg_cases.head(30)

In [ ]:
#sum of 3 years
district_yearly_avg_cases = (
    district_monthly_avg_cases
    .groupby('District')['Cases']
    .mean()
    .reset_index())

district_yearly_avg_cases


In [ ]:
#province

In [ ]:
# Get the unique District to Province mapping from the original 'df' DataFrame
district_province_map = df[['District', 'Province']].drop_duplicates()

# Merge this mapping into district_year_cases to add the 'Province' column
district_year_cases_with_province = pd.merge(
    district_yearly_avg_cases,
    district_province_map,
    on='District',
    how='left'
)
district_year_cases_with_province.head()


In [ ]:

# Now, group by 'Province' using the DataFrame that includes 'Province'
province_total_cases = (
    district_year_cases_with_province
    .groupby('Province')['Cases']
    .sum()
    .reset_index(name='total_cases')
)
province_total_cases

In [ ]:
population = {
    'Western': 6149000,
    'Central': 2766000,
    'Southern': 2654000,
    'Northern': 1143000,
    'Eastern': 1729000,
    'North Western': 2551000,
    'North central': 1377000,
    'Uva': 1376000,
    'Sabaragamuwa': 2058000
}


In [ ]:
province_total_cases['Population'] = (
    province_total_cases['Province'].map(population)
)
province_total_cases



In [ ]:
province_total_cases['incidence_per_100k'] = (
    province_total_cases['total_cases'] /
    province_total_cases['Population']
) * 100000
province_total_cases


In [ ]:
low_threshold = province_total_cases['incidence_per_100k'].quantile(0.33)
high_threshold = province_total_cases['incidence_per_100k'].quantile(0.66)

low_threshold, high_threshold

def classify_risk(incidence):
    if incidence <= low_threshold:
        return 'Low Risk'
    elif incidence <= high_threshold:
        return 'Medium Risk'
    else:
        return 'High Risk'

province_total_cases['risk_level'] = (
    province_total_cases['incidence_per_100k']
    .apply(classify_risk)
)

province_total_cases[
    ['Province', 'incidence_per_100k', 'risk_level']
]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot 1: Incidence per 100k people by Province
plt.figure(figsize=(12, 7))
sns.barplot(x=province_total_cases['Province'],
y=province_total_cases['incidence_per_100k'],
palette='viridis')
plt.title('Dengue Incidence per 100,000 People by Province')
plt.xlabel('Province')
plt.ylabel('Incidence per 100,000 People')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#hypothesis testsing

In [ ]:
from scipy.stats import mannwhitneyu

before_2020 = df[df['Year'] < 2020]['Cases']
after_2020 = df[df['Year'] >= 2020]['Cases']

stat, p_value = mannwhitneyu(before_2020, after_2020, alternative='two-sided')

print("Mann-Whitney U statistic:", stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Significant difference in dengue cases before and after 2020.")
else:
    print("No significant difference in dengue cases before and after 2020.")

In [ ]:
#TIME SERIES ANALYSIS

In [ ]:
import pandas as pd

# Create proper datetime column
df['Date'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY=1))

# Sort by date
df = df.sort_values('Date')
colombo_data = df[df['District'] == 'Colombo']
colombo_ts = colombo_data.groupby('Date')['Cases'].sum()

import matplotlib.pyplot as plt

plt.figure()
plt.plot(colombo_ts)


plt.title("Colombo District - Monthly Dengue Cases")
plt.xlabel("Time")
plt.ylabel("Number of Cases")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(colombo_ts)

print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])

alpha = 0.05
if adf_result[1] < alpha:
    print("Series is Stationary")
else:
    print("Series is Non-Stationary (Trend exists)")

In [ ]:
colombo_diff = colombo_ts.diff().dropna()


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_result_diff = adfuller(colombo_diff)

print("ADF Statistic:", adf_result_diff[0])
print("p-value:", adf_result_diff[1])

if adf_result_diff[1] < 0.05:
    print("Differenced series is Stationary")
else:
    print("Still Non-Stationary")

In [ ]:
seasonal_diff = colombo_ts.diff(periods=12).dropna()


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_result_diff = adfuller(seasonal_diff)

print("ADF Statistic:", adf_result_diff[0])
print("p-value:", adf_result_diff[1])

if adf_result_diff[1] < 0.05:
    print("Differenced series is Stationary")
else:
    print("Still Non-Stationary")

In [ ]:
colombo_diff2 = colombo_diff.diff().dropna()

from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(colombo_diff2)
print(adf_result[0])
print(adf_result[1])

In [ ]:
# Step 3: Plot ACF and PACF on stationary data
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(colombo_diff2, lags=15, ax=axes[0])
plot_pacf(colombo_diff2, lags=15, ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import itertools
import warnings
warnings.filterwarnings('ignore')

# Try different p and q combinations
p_values = [0, 1, 2]
q_values = [0, 1, 2]
d = 2

best_aic = float('inf')
best_order = None

for p, q in itertools.product(p_values, q_values):
    try:
        model = ARIMA(colombo_ts, order=(p, d, q))
        result = model.fit()
        print(f"ARIMA({p},{d},{q}) - AIC: {result.aic:.2f}")
        if result.aic < best_aic:
            best_aic = result.aic
            best_order = (p, d, q)
    except:
        continue

print(f"\nBest Model: ARIMA{best_order} with AIC: {best_aic:.2f}")

In [ ]:
# Fit the best model
best_model = ARIMA(colombo_ts, order=(0, 2, 2))
best_result = best_model.fit()
print(best_result.summary())

# Step 1: Residual Diagnostics
best_result.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()

# Step 2: Ljung-Box Test (residuals should be white noise)
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_test = acorr_ljungbox(best_result.resid, lags=[10], return_df=True)
print("\nLjung-Box Test:")
print(lb_test)
# p-value > 0.05 means residuals are white noise (good!)

# Forecast only 6 months
forecast = best_result.get_forecast(steps=6)
forecast_mean = forecast.predicted_mean.clip(lower=0)
forecast_ci = forecast.conf_int()
forecast_ci[forecast_ci < 0] = 0  # clip negatives

print("6-Month Forecast:")
print(forecast_mean.round().astype(int))

# Plot forecast
plt.figure(figsize=(12, 5))
plt.plot(colombo_ts, label='Observed')
plt.plot(forecast_mean, label='Forecast', color='red')
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0],
                 forecast_ci.iloc[:, 1],
                 color='red', alpha=0.2, label='95% Confidence Interval')
plt.title("Colombo Dengue Cases - ARIMA(0,2,2) Forecast")
plt.xlabel("Date")
plt.ylabel("Number of Cases")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
'''
Forecast interpretation:

The model predicts dengue cases will continue at around 3,300–3,600 cases per month from early to mid 2022
The trend is slightly increasing, which aligns with the upward pattern visible at the end of 2021

'''

### Evaluating Forecast Accuracy with Train-Test Split (MAE & RMSE)



In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Define the size of the test set (e.g., last 6 months, matching the forecast steps)
n_test = 6

# Split the data into training and testing sets
train_data = colombo_ts[:-n_test]
test_data = colombo_ts[-n_test:]

print(f"Training data period: {train_data.index.min()} to {train_data.index.max()}")
print(f"Testing data period: {test_data.index.min()} to {test_data.index.max()}")
print(f"Number of training points: {len(train_data)}")
print(f"Number of testing points: {len(test_data)}")

Re-train the ARIMA model

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Re-fit the best model on the training data
eval_model = ARIMA(train_data, order=best_order) # best_order was (0,2,2)
eval_result = eval_model.fit()

print("Model re-trained on training data.")

# Generate a forecast for the test period
forecast_eval = eval_result.get_forecast(steps=n_test)
forecast_mean_eval = forecast_eval.predicted_mean.clip(lower=0)

print("\nForecasted values for the test period:")
print(forecast_mean_eval.round().astype(int))

print("\nActual values for the test period:")
print(test_data.round().astype(int))

Finally, we can calculate the Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) to quantify the accuracy of our forecast on the held-out test data.

In [ ]:
# Calculate MAE and RMSE
mae = mean_absolute_error(test_data, forecast_mean_eval)
rmse = np.sqrt(mean_squared_error(test_data, forecast_mean_eval))

print(f"Mean Absolute Error (MAE) on test set: {mae:.2f}")
print(f"Root Mean Squared Error (RMSE) on test set: {rmse:.2f}")

# Plot the actual vs. forecasted values for the test set
plt.figure(figsize=(12, 6))
plt.plot(train_data.index, train_data, label='Training Data', color='blue')
plt.plot(test_data.index, test_data, label='Actual Test Data', color='green')
plt.plot(forecast_mean_eval.index, forecast_mean_eval, label='Forecasted Test Data', color='red', linestyle='--')
plt.title('ARIMA Forecast vs. Actuals on Test Set')
plt.xlabel('Date')
plt.ylabel('Number of Cases')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()